# Ładowanie Bibliotek


In [133]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score
from sklearn import datasets
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler

sns.set()

# Ładowanie danych

In [134]:
df = pd.read_csv('creditcard.csv', na_values=['?'])
df_test = pd.read_csv('creditcard_test.csv', na_values=['?'])

# Praca na danych

In [135]:
df.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
0,b,27.25,0.625,u,g,aa,v,0.455,t,f,0,t,g,200.0,0,-
1,b,28.92,15.000,u,g,c,h,5.335,t,t,11,f,g,0.0,2283,+
2,b,47.17,5.835,u,g,w,v,5.500,f,f,0,f,g,465.0,150,-
3,b,23.08,2.500,u,g,ff,ff,0.085,f,f,0,t,g,100.0,4208,-
4,b,32.67,9.000,y,p,w,h,5.250,t,f,0,t,g,154.0,0,+


In [136]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A1      629 non-null    str    
 1   A2      628 non-null    float64
 2   A3      640 non-null    float64
 3   A4      634 non-null    str    
 4   A5      634 non-null    str    
 5   A6      631 non-null    str    
 6   A7      631 non-null    str    
 7   A8      640 non-null    float64
 8   A9      640 non-null    str    
 9   A10     640 non-null    str    
 10  A11     640 non-null    int64  
 11  A12     640 non-null    str    
 12  A13     640 non-null    str    
 13  A14     628 non-null    float64
 14  A15     640 non-null    int64  
 15  A16     640 non-null    str    
dtypes: float64(4), int64(2), str(10)
memory usage: 80.1 KB


In [137]:
ciagle = ['A2', 'A3', 'A8', 'A11', 'A14', 'A15']
kategoryczne = []

for col in df.columns:
    if col not in ciagle and col != 'A16':
        kategoryczne.append(col)

for col in kategoryczne:
    unique = df[col].unique()
    print(col)
    print(unique)

A1
<StringArray>
['b', 'a', nan]
Length: 3, dtype: str
A4
<StringArray>
['u', 'y', nan, 'l']
Length: 4, dtype: str
A5
<StringArray>
['g', 'p', nan, 'gg']
Length: 4, dtype: str
A6
<StringArray>
['aa', 'c', 'w', 'ff', 'e', 'd', 'q', 'j', 'k', 'i', 'cc', 'x', 'm', nan, 'r']
Length: 15, dtype: str
A7
<StringArray>
['v', 'h', 'ff', 'bb', 'dd', 'j', nan, 'n', 'z', 'o']
Length: 10, dtype: str
A9
<StringArray>
['t', 'f']
Length: 2, dtype: str
A10
<StringArray>
['f', 't']
Length: 2, dtype: str
A12
<StringArray>
['t', 'f']
Length: 2, dtype: str
A13
<StringArray>
['g', 's', 'p']
Length: 3, dtype: str


In [138]:
df['A2'] = df['A2'].astype(float)
df['A14'] = df['A14'].astype(float)

df_test['A2'] = df_test['A2'].astype(float)
df_test['A14'] = df_test['A14'].astype(float)


for col in ciagle:
    df[col] = df[col].fillna(df[col].median())
    df_test[col] = df_test[col].fillna(df_test[col].median())


for col in kategoryczne:
    df[col] = df[col].fillna(df[col].mode()[0])
    df_test[col] = df_test[col].fillna(df_test[col].mode()[0])


binarne_mapowanie = {'t': 1, 'f': 0}
df['A9'] = df['A9'].map(binarne_mapowanie)
df['A10'] = df['A10'].map(binarne_mapowanie)
df['A12'] = df['A12'].map(binarne_mapowanie)

df_test['A9'] = df_test['A9'].map(binarne_mapowanie)
df_test['A10'] = df_test['A10'].map(binarne_mapowanie)
df_test['A12'] = df_test['A12'].map(binarne_mapowanie)


df['A16'] = df['A16'].map({'+': 1, '-': 0})


kolumny_do_dummies = ['A1', 'A4', 'A5', 'A6', 'A7', 'A13']
df = pd.get_dummies(df, columns=kolumny_do_dummies, drop_first=True)
df_test = pd.get_dummies(df_test, columns=kolumny_do_dummies, drop_first=True)

In [139]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 38 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A2      640 non-null    float64
 1   A3      640 non-null    float64
 2   A8      640 non-null    float64
 3   A9      640 non-null    int64  
 4   A10     640 non-null    int64  
 5   A11     640 non-null    int64  
 6   A12     640 non-null    int64  
 7   A14     640 non-null    float64
 8   A15     640 non-null    int64  
 9   A16     640 non-null    int64  
 10  A1_b    640 non-null    bool   
 11  A4_u    640 non-null    bool   
 12  A4_y    640 non-null    bool   
 13  A5_gg   640 non-null    bool   
 14  A5_p    640 non-null    bool   
 15  A6_c    640 non-null    bool   
 16  A6_cc   640 non-null    bool   
 17  A6_d    640 non-null    bool   
 18  A6_e    640 non-null    bool   
 19  A6_ff   640 non-null    bool   
 20  A6_i    640 non-null    bool   
 21  A6_j    640 non-null    bool   
 22  A6_k    640 n

In [140]:
for col in df.columns:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)
for col in df_test.columns:
    if df_test[col].dtype == bool:
        df_test[col] = df_test[col].astype(int)

In [141]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 640 entries, 0 to 639
Data columns (total 38 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A2      640 non-null    float64
 1   A3      640 non-null    float64
 2   A8      640 non-null    float64
 3   A9      640 non-null    int64  
 4   A10     640 non-null    int64  
 5   A11     640 non-null    int64  
 6   A12     640 non-null    int64  
 7   A14     640 non-null    float64
 8   A15     640 non-null    int64  
 9   A16     640 non-null    int64  
 10  A1_b    640 non-null    int64  
 11  A4_u    640 non-null    int64  
 12  A4_y    640 non-null    int64  
 13  A5_gg   640 non-null    int64  
 14  A5_p    640 non-null    int64  
 15  A6_c    640 non-null    int64  
 16  A6_cc   640 non-null    int64  
 17  A6_d    640 non-null    int64  
 18  A6_e    640 non-null    int64  
 19  A6_ff   640 non-null    int64  
 20  A6_i    640 non-null    int64  
 21  A6_j    640 non-null    int64  
 22  A6_k    640 n

# Budowanie modelu

In [142]:
X = df.drop(columns='A16') 
y = df['A16']  

In [143]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [144]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [145]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}


log_reg = LogisticRegression(max_iter=500, random_state=0)
grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...andom_state=0)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.001, 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate para

In [146]:
reg_log = grid_search.best_estimator_

In [147]:
y_pred_proba = reg_log.predict_proba(X_test_scaled)[:, 1]

In [148]:
best_threshold = 0.5
max_profit = -float('inf')

In [149]:
for threshold in np.arange(0.05, 0.95, 0.05):
    y_pred_custom = (y_pred_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_custom)
    TN, FP, FN, TP = cm.ravel()
    profit = (25000 * TP) - (10000 * FP)
    if profit > max_profit:
        max_profit = profit
        best_threshold = threshold

In [150]:
print(best_threshold)
print(max_profit)

0.35000000000000003
1105000


In [151]:
y_test_optimal_pred = (y_pred_proba >= best_threshold).astype(int)

In [152]:
confusion_matrix(y_test, y_test_optimal_pred)

array([[65, 12],
       [ 2, 49]])

In [153]:
accuracy_score(y_test, y_test_optimal_pred)

0.890625

In [154]:
recall_score(y_test, y_test_optimal_pred)

0.9607843137254902

# Wyniki


In [155]:
df_test = df_test.reindex(columns=X.columns, fill_value=0)

In [156]:
X_final_test_scaled = scaler.transform(df_test)
final_proba = reg_log.predict_proba(X_final_test_scaled)[:, 1]
final_predictions = (final_proba >= best_threshold).astype(int)

In [157]:
final_predictions_labels = np.where(final_predictions == 1, '+', '-')

In [158]:
submission = pd.DataFrame({'A16': final_predictions_labels})
submission.to_csv('k.csv', index=False)